In [1]:

import asyncio
import warnings
import pandas as pd
from pathlib import Path

# Парсеры
from antimony import antimony_parser
from westmetall import westmetall_async
from lme import lme_selenium_async
from lbma import lbma_prescious_async
from cbr import cb_currency, cb_metalls
from nbk import nbk_tenge_async
from shmet import shmet_optimized_async
from kitco import kitco_parser_async

# Сервисные функции из service_layer
from service_layer import (
    read_db,
    save_db,
    check_df,
    check_and_save_pair,
    show_db,
    excel_to_csv_db
)

warnings.filterwarnings("ignore")


# ================================================================
# Пути к базам (реальная структура проекта)
# ================================================================
LME_PATH = Path("lme/data/LME_db_new.xlsx")
WESTMETALL_PATH = Path("westmetall/data/LME_westmetall_db.xlsx")

KITCO_PATH = Path("kitco/data/kitko_db.xlsx")
LBMA_PATH = Path("lbma/data/lbma_kitco_subs.xlsx")

ANTIMONY_PATH = Path("antimony/data/antimony.xlsx")

CB_CURRENCY_PATH = Path("cbr/data/cb_currency.xlsx")
CB_METALLS_PATH = Path("cbr/data/cb_metalls.xlsx")

NBK_PATH = Path("nbk/data/nbk_tenge.xlsx")
SHMET_PATH = Path("shmet/data/shmet_historical.xlsx")


# ================================================================
# Проверка целостности парных баз
# ================================================================
def db_check():
    """
    Проверка целостности парных баз с очисткой дубликатов
    и сохранением первого (более раннего) вхождения.
    """
    print("Проверка LME / Westmetall...")
    check_and_save_pair(
        LME_PATH,
        WESTMETALL_PATH,
        pair_name="LME / Westmetall",
        index=False,
    )

    print("Проверка Kitco / LBMA...")
    check_and_save_pair(
        KITCO_PATH,
        LBMA_PATH,
        pair_name="Kitco / LBMA",
        index=False,
    )
    
async def main():
    print("Parsing started...")

    tasks = {
        "lme": lme_selenium_async(),
        "antimony": antimony_parser(),
        "westmetall": westmetall_async(),
        "lbma": lbma_prescious_async(),
        "kitco": kitco_parser_async(),
        "cb_currency": cb_currency(),
        "cb_metalls": cb_metalls(),
        "nbk": nbk_tenge_async(),
        "shmet": shmet_optimized_async(),
    }

    # return_exceptions=True — чтобы падение одного парсера
    # не останавливало остальные
    results = await asyncio.gather(
        *tasks.values(),
        return_exceptions=True,
    )

    for name, result in zip(tasks.keys(), results):
        if isinstance(result, Exception):
            print(f"❌ Ошибка в {name}: {result}")

    print("All tasks are done!")
    print("+" * 64)
    print("Checking DB...")

    db_check()

    print("DB check completed!")
    print("+" * 64)
    print("Visual control")
    print("+" * 64)
    
    print("Converting Excel to CSV...")
    excel_to_csv_db()
    print("CSV conversion completed!")

    # Базовые металлы
    show_db("lme_selenium_db", LME_PATH, sheet_name=0)
    show_db("westmetall_db", WESTMETALL_PATH, sheet_name=0)

    # Драгоценные металлы
    show_db("kitco_db", KITCO_PATH, sheet_name=0)
    show_db("lbma_precious_db", LBMA_PATH, sheet_name=0)

    # Антимоний
    show_db("antimony_db", ANTIMONY_PATH, sheet_name=0)

    # ЦБ РФ: валюты (каждая на своем листе)
    for currency in [
        "USD",
        "EUR",
        "British_Pound",
        "China_Yuan",
        "Japanese_Yen",
        "Swiss_Franc",
    ]:
        show_db(
            f"cb_currency ({currency})",
            CB_CURRENCY_PATH,
            sheet_name=currency,
        )

    # ЦБ РФ: металлы
    show_db("cb_metalls_db", CB_METALLS_PATH, sheet_name=0)

    # Казахстан и SHMET
    show_db("nbk_tenge_db", NBK_PATH, sheet_name=0)
    show_db("shmet_historical_db", SHMET_PATH, sheet_name=0, show_head=True)


# Для Jupyter используем await, а не asyncio.run()
await main()

Parsing started...
🚀 LME parsing started...
antimony parsing is DONE
NBK_tenge parsing is DONE! (1698 строк)
CB_metalls parsing is DONE!
WESTMETALL is done!!!
USD is done!
EUR is done!
Australian_Dollar is done!
China_Yuan is done!
British_Pound is done!
Kazakhstan_Tenge is done!
Japanese_Yen is done!
Swiss_Franc is done!
CB_currency parsing is DONE!
Произошла ошибка KITCO: Не найдены блоки <div class='grid'> на странице Kitco
SHMET is done!!!
LBMA is done!!!
✅ LME_main is done!!!
All tasks are done!
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Checking DB...
Проверка LME / Westmetall...
LME / Westmetall: добавлены пропущенные даты и удалены дубликаты
Проверка Kitco / LBMA...
Kitco / LBMA: добавлены пропущенные даты и удалены дубликаты
DB check completed!
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Visual control
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Converting Excel to CSV...
Поиск Excel файлов...
Найдено 9 Excel файл

,date,aluminium,copper,lead,nickel,zink,tin
1194,2026-09-21,3262.5,14788.0,1890.0,16125,4018.0,53785
1195,2026-09-22,3242.0,14797.0,1909.0,16410,4006.0,53745
1196,2026-09-23,3240.0,14735.0,1885.5,16430,3949.0,54050
1197,2026-09-24,3228.0,14765.0,1900.0,16320,3981.0,53875
1198,2026-09-25,3254.0,14740.0,1902.5,16050,4060.0,54025


westmetall_db


,date,aluminium,copper,lead,nickel,zink,tin
1194,2026-09-21,3262.5,14788.0,1890.0,16125,4018.0,53785
1195,2026-09-22,3242.0,14797.0,1909.0,16410,4006.0,53745
1196,2026-09-23,3240.0,14735.0,1885.5,16430,3949.0,54050
1197,2026-09-24,3228.0,14765.0,1900.0,16320,3981.0,53875
1198,2026-09-25,3254.0,14740.0,1902.5,16050,4060.0,54025


kitco_db


...

,date,gold,silver,platinum,palladium
917,2026-09-22,11756.21,181.24,4848.32,3556.75
918,2026-09-23,11687.47,178.05,4933.64,3588.88
919,2026-09-24,11747.89,178.10,4922.42,3575.34
920,2026-09-25,11695.60,177.94,4826.52,3472.96
921,2026-09-26,11568.93,172.41,4755.13,3443.10


nbk_tenge_db


,date,Числовое значение,ДОЛЛАР США
1693,2026-09-22,1,448.44
1694,2026-09-23,1,447.85
1695,2026-09-24,1,445.65
1696,2026-09-25,1,441.89
1697,2026-09-26,1,441.88


shmet_historical_db


,date,price,unit
0,2020-01-10,48605,Yuan/MT
1,2020-01-14,48990,Yuan/MT
2,2020-01-15,49060,Yuan/MT
3,2020-01-16,48950,Yuan/MT
4,2020-01-17,48930,Yuan/MT
